In [3]:
###Secrets & Installation

#turns off any warnings
import warnings
warnings.filterwarnings('ignore')

#various modules
import requests 
import pandas as pd
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from lazypredict.Supervised import LazyRegressor
from sklearn import datasets
from sklearn.utils import shuffle
from pytrends.request import TrendReq
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV
from sklearn.linear_model import OrthogonalMatchingPursuitCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, roc_auc_score
from sklearn.calibration import calibration_curve

In [ ]:
###Stadium Details

In [4]:
#Stadium API and Staging
#API to pull in stadium details

apiKey_sportsdata = "94fac3754c28473bbebd232bc8164e37"
url = "https://api.sportsdata.io/api/nfl/odds/json/Stadiums" 

#put in your own key
headers = { 
    "Ocp-Apim-Subscription-Key": apiKey_sportsdata
}

response = requests.get(url, headers=headers) 
stadiums = response.json() 

#pulling in data
df1 = pd.DataFrame(stadiums) 

#dropping unneeded data
df1.drop(columns=['City', 'State', 'Country', 'Capacity', 'GeoLat', 'GeoLong'], inplace=True)

#applying numeric details for playing  surface and type
df1['PlayingSurface'] = df1['PlayingSurface'].replace({'Artificial': 1, 'Dome': 2, 'Grass': 3})
df1['Type'] = df1['Type'].replace({'Dome': 1, 'Outdoor': 2, 'RetractableDome': 3})

#renaming one of the columns
df1.rename(columns = {'Name':'Stadium'}, inplace = True)
df1.head()

,StadiumID,Stadium,PlayingSurface,Type
0,1,Highmark Stadium,1.00,2
1,2,Hard Rock Stadium,3.00,2
2,3,MetLife Stadium,1.00,2
3,4,Gillette Stadium,1.00,2
4,5,Paycor Stadium,1.00,2


In [ ]:
# Save df1 as a pickle file
with open('df1.pickle', 'wb') as f:
    pickle.dump(df1, f)

In [ ]:
# Load df1 from the pickle file
with open('df1.pickle', 'rb') as f:
    df1 = pickle.load(f)

In [6]:
#bringing in regular season 2023 and 2024 data
teamstats = pd.DataFrame()
for i in range(1,19):
    url10 = f"https://api.sportsdata.io/api/nfl/odds/json/TeamGameStats/2022REG/{i}"
    url11 = f"https://api.sportsdata.io/api/nfl/odds/json/TeamGameStats/2023REG/{i}"
    url12 = f"https://api.sportsdata.io/api/nfl/odds/json/TeamGameStats/2024REG/{i}"

    headers = {
        "Ocp-Apim-Subscription-Key": apiKey_sportsdata
    }

    response10 = requests.get(url10, headers=headers) 
    response11 = requests.get(url11, headers=headers)
    response12 = requests.get(url12, headers=headers)

    bettingdata10 = response10.json() 
    bettingdata11 = response11.json()
    bettingdata12 = response12.json()

    e10 = pd.DataFrame(bettingdata10)
    e11 = pd.DataFrame(bettingdata11)
    e12 = pd.DataFrame(bettingdata12)

    teamstats = pd.concat([teamstats,e10,e11,e12], ignore_index = True)

#bringing in regular season 2020, 2021 and 2022 data postseason
for i in range(1,5):
    url13 = f"https://api.sportsdata.io/api/nfl/odds/json/TeamGameStats/2022POST/{i}"
    url14 = f"https://api.sportsdata.io/api/nfl/odds/json/TeamGameStats/2023POST/{i}"
    url15 = f"https://api.sportsdata.io/api/nfl/odds/json/TeamGameStats/2024POST/{i}"

    headers = {
        "Ocp-Apim-Subscription-Key": apiKey_sportsdata
    }

    response13 = requests.get(url13, headers=headers) 
    response14 = requests.get(url14, headers=headers) 
    response15 = requests.get(url15, headers=headers) 

    bettingdata13 = response13.json() 
    bettingdata14 = response14.json()
    bettingdata15 = response15.json()

    e13 = pd.DataFrame(bettingdata13)
    e14 = pd.DataFrame(bettingdata14)
    e15 = pd.DataFrame(bettingdata15)

    teamstats = pd.concat([teamstats,e13,e14, e15], ignore_index = True)

#dropping rows with NA's (ie bills/bengals game)
teamstats.dropna(subset=['OpponentScore', 'TotalScore'],inplace=True)

#changing some of the stadium detail as names in the stadium section are current
teamstats['Stadium'] = teamstats['Stadium'].replace({'Paul Brown Stadium':'Paycor Stadium','Mercedes-Benz Superdome':'Caesars Superdome','Heinz Field':'Acrisure Stadium','CenturyLink Field':'Lumen Field','Bills Stadium':'Highmark Stadium'})

#changing the date time column to date format allowing to create time based columns
teamstats['Date'] = pd.to_datetime(teamstats['Date'])

#adding time features
teamstats['month'] = teamstats['Date'].dt.month
teamstats['year'] = teamstats['Date'].dt.year
teamstats['dayofyear'] = teamstats['Date'].dt.dayofyear

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


ValueError: If using all scalar values, you must pass an index

In [ ]:
# Save teamstats as a pickle file
with open('teamstats.pickle', 'wb') as f:
    pickle.dump(teamstats, f)

In [ ]:
# Load teamstats from the pickle file
with open('teamstats.pickle', 'rb') as f:
    teamstats = pickle.load(f)

In [ ]:
teamstats.head()

In [ ]:
# Get the column names
col_names = teamstats.columns

# Create a DataFrame from the column names
df_col_names = pd.DataFrame(col_names, columns=['Column Names'])

# Set maximum rows to display to None
pd.set_option('display.max_rows', None)

# Reset display option to default
#pd.reset_option('display.max_rows')

# Display the DataFrame
df_col_names

#took out RedZoneConversions - left in RedZone Percantage

In [ ]:
#bringing in regular season 2021 and 2022 data
bettingdata2022 = pd.DataFrame()
for i in range(1,19):
    url = f"https://api.sportsdata.io/api/nfl/odds/json/GameOddsByWeek/2022/{i}"
    url3 = f"https://api.sportsdata.io/api/nfl/odds/json/GameOddsByWeek/2023/{i}"
    url5 = f"https://api.sportsdata.io/api/nfl/odds/json/GameOddsByWeek/2024/{i}"

    headers = {
        "Ocp-Apim-Subscription-Key": apiKey3
    }

    response1 = requests.get(url, headers=headers) 
    response3 = requests.get(url3, headers=headers)
    response5 = requests.get(url5, headers=headers)    

    bettingdata1 = response1.json() 
    bettingdata3 = response3.json()
    bettingdata5 = response5.json()

    d1 = pd.DataFrame(bettingdata1)
    d3 = pd.DataFrame(bettingdata3)
    d5 = pd.DataFrame(bettingdata5)

    bettingdata2024 = pd.concat([bettingdata2022,d1,d2,d5], ignore_index = True)

    #brining in 2022POST, 2023POST and 2024POST data
for i in range(1,5):
    url2 = f"https://api.sportsdata.io/api/nfl/odds/json/GameOddsByWeek/2022POST/{i}"
    url4 = f"https://api.sportsdata.io/api/nfl/odds/json/GameOddsByWeek/2023POST/{i}"
    url6 = f"https://api.sportsdata.io/api/nfl/odds/json/GameOddsByWeek/2024POST/{i}"

    headers = {
        "Ocp-Apim-Subscription-Key": apiKey3
    }

    response2 = requests.get(url2, headers=headers) 
    response4 = requests.get(url4, headers=headers) 
    response6 = requests.get(url6, headers=headers)

    bettingdata2 = response2.json() 
    bettingdata4 = response4.json() 
    bettingdata6 = response6.json() 

    d2 = pd.DataFrame(bettingdata2)
    d4 = pd.DataFrame(bettingdata4)
    d6 = pd.DataFrame(bettingdata6)

    df= pd.concat([bettingdata2022, d2,d4, d6], ignore_index = True)

# Split the PregameOdds column into separate rows
bettingdata2024master = df.explode('PregameOdds')

#making the data a string so the next step can take place
df['PregameOdds'] = df['PregameOdds'].map(str)

#bettingdata2022master['GameOddId'] = bettingdata2022master.PregameOdds.str.split(',', expand=True)[0].str.split(':', expand=True)[1].str.strip()
df['GameOddId'] = df.PregameOdds.str.split(',', expand=True)[0].str.split(':', expand=True)[1].str.strip()
df['Sportsbook'] = df.PregameOdds.str.split(',', expand=True)[1].str.split(':', expand=True)[1].str.strip()
df['ScoreId'] = df.PregameOdds.str.split(',', expand=True)[2].str.split(':', expand=True)[1].str.strip()
df['Created'] = df.PregameOdds.str.split(',', expand=True)[3].str.split(':', expand=True)[1].str.strip()
df['Updated'] = df.PregameOdds.str.split(',', expand=True)[4].str.split(':', expand=True)[1].str.strip()
df['HomeMoneyLine'] = df.PregameOdds.str.split(',', expand=True)[5].str.split(':', expand=True)[1].str.strip()
df['AwayMoneyLine'] = df.PregameOdds.str.split(',', expand=True)[6].str.split(':', expand=True)[1].str.strip()
df['DrawMoneyLine'] = df.PregameOdds.str.split(',', expand=True)[7].str.split(':', expand=True)[1].str.strip()
df['HomePointSpread'] = df.PregameOdds.str.split(',', expand=True)[8].str.split(':', expand=True)[1].str.strip()
df['AwayPointSpread'] = df.PregameOdds.str.split(',', expand=True)[9].str.split(':', expand=True)[1].str.strip()
df['HomePointSpreadPayout'] = df.PregameOdds.str.split(',', expand=True)[10].str.split(':', expand=True)[1].str.strip()
df['AwayPointSpreadPayout'] = df.PregameOdds.str.split(',', expand=True)[11].str.split(':', expand=True)[1].str.strip()
df['OverUnder'] = df.PregameOdds.str.split(',', expand=True)[12].str.split(':', expand=True)[1].str.strip()
df['OverPayout'] = df.PregameOdds.str.split(',', expand=True)[13].str.split(':', expand=True)[1].str.strip()
df['UnderPayout'] = df.PregameOdds.str.split(',', expand=True)[14].str.split(':', expand=True)[1].str.strip()
df['SportsbookId'] = df.PregameOdds.str.split(',', expand=True)[15].str.split(':', expand=True)[1].str.strip()

#changing the date time column to date format allowing to create time based columns
df['DateTime'] = pd.to_datetime(df['DateTime'])

#adding time features
df['hour'] = df['DateTime'].dt.hour
df['dayofweek'] = df['DateTime'].dt.dayofweek
df['quarter'] = df['DateTime'].dt.quarter
df['month'] = df['DateTime'].dt.month
df['year'] = df['DateTime'].dt.year
df['dayofyear'] = df['DateTime'].dt.dayofyear
df['dayofmonth'] = df['DateTime'].dt.day
df['weekofyear'] = df['DateTime'].dt.weekofyear

#dropping rows with NA's (future data and bills/bengals game)
df.dropna(subset=['HomeTeamScore', 'AwayTeamScore'],inplace=True)

#changing numeric data to a float
df['HomeMoneyLine'] = df['HomeMoneyLine'].astype(float)
df['AwayMoneyLine'] = df['AwayMoneyLine'].astype(float)
df['HomePointSpread'] = df['HomePointSpread'].astype(float)
df['AwayPointSpread'] = df['AwayPointSpread'].astype(float)
df['HomePointSpreadPayout'] = df['HomePointSpreadPayout'].astype(float)
df['AwayPointSpreadPayout'] = df['AwayPointSpreadPayout'].astype(float)
df['OverUnder'] = df['OverUnder'].astype(float)
df['OverPayout'] = df['OverPayout'].astype(float)
df['UnderPayout'] = df['UnderPayout'].astype(float)

In [ ]:
# Save df as a pickle file
with open('df.pickle', 'wb') as f:
    pickle.dump(df, f)

In [ ]:
# Load df from the pickle file
with open('df.pickle', 'rb') as f:
    df = pickle.load(f)

In [ ]:
df.head()

In [ ]:
# Get the column names
col_names = df.columns

# Create a DataFrame from the column names
df_col_names = pd.DataFrame(col_names, columns=['Column Names'])

# Set maximum rows to display to None
pd.set_option('display.max_rows', None)

# Reset display option to default
#pd.reset_option('display.max_rows')

# Display the DataFrame
df_col_names

#took out RedZoneConversions - left in RedZone Percantage

In [10]:
# The initial setup loop is just to show the classes and doesn't save the data.
# The `response` and `soup` variables from this block will be overwritten
# by the main scraping loop, but it's okay to keep for debugging table classes.
print('Starting initial class check...')
# Initial request block (Keep for reference/debugging, but not strictly needed for the final DataFrame)
for year in [2022, 2023, 2024]:
    # Note: 'Alan-Eck' is in the first list but not the second, and 'Jerome-Boger' is only in the second.
    # We will use the second list for the main scraping loop.
    for ref in ['Brad-Allen', 'Tra-Blake', 'Clete-Blakeman', 'Alan-Eck', 'Jerome-Boger', 'Carl-Cheffers', 'Land-Clark', 'Adrian-Hill', 'Shawn-Hochuli', 'John-Hussey', 'Alex-Kemp', 'Clay-Martin', 'Scott-Novak', 'Brad-Rogers', 'Shawn-Smith', 'Ron-Torbert', 'Bill-Vinovich', 'Craig-Wrolstad']:
        url = "https://www.nflpenalties.com/referee/" + ref + "?year=" + str(year)
response = requests.get(url) # This will only execute for the LAST ref/year in the loop above
soup = BeautifulSoup(response.text, 'html.parser')

print('Classes of each table:')
for table in soup.find_all('table'):
    print(table.get('class'))
print('---')
# End of initial check

# 💡 FIX: Use a list of dictionaries to collect data, then use pd.concat() once.
data_to_append = []

#loop that goes through each referee games ref'd for 2022, 2023 and 2024
referee_list = ['Brad-Allen', 'Tra-Blake', 'Clete-Blakeman', 'Alan-Eck', 'Jerome-Boger', 'Carl-Cheffers', 'Land-Clark', 'Adrian-Hill', 'Shawn-Hochuli', 'John-Hussey', 'Alex-Kemp', 'Clay-Martin', 'Scott-Novak', 'Brad-Rogers', 'Shawn-Smith', 'Ron-Torbert', 'Bill-Vinovich', 'Craig-Wrolstad']

for ref in referee_list:
    print(f"Scraping data for referee: {ref}")
    for year in [2022, 2023, 2024]:
        url = "https://www.nflpenalties.com/referee/" + ref + "?year=" + str(year)
        response = requests.get(url)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            # Assuming 'footable' is the correct table class
            table = soup.find('table',{'class':'footable'})
            
            # Check if the table was found
            if table is None:
                print(f"Warning: Table 'footable' not found for {ref} in {year}")
                continue
                
            rows = table.find_all('tr')

            for row in rows:
                cols = row.find_all('td')
                
                # The data rows have 4 columns (Date, Week, Team, AwayTeam)
                if len(cols) >= 4:
                    date = cols[0].text.strip()
                    week = cols[1].text.strip()
                    team = cols[2].text.strip()
                    awayteam = cols[3].text.strip()
                    
                    # Ensure we skip the 'Totals' row which will be in the date column
                    if date != 'Totals':
                        # Append a dictionary for the new row to the list
                        data_to_append.append({
                            'Date': date, 
                            'Week': week, 
                            'FullName': team, 
                            'AwayTeam': awayteam, 
                            'Referee': ref
                        })
        else:
            print(f"Request failed for {ref} in {year} with status code: {response.status_code}")
            continue

# Create the initial DataFrame from the collected list of dictionaries
df7000 = pd.DataFrame(data_to_append)
print('---')
print(f"Data scraped. Initial DataFrame size: {len(df7000)} rows.")

## Merging and Feature Engineering (No changes needed here)

#creating a data frame to expand on team name based on how data is pulled in above
df50 = pd.DataFrame({'Team': ['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 
                             'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'MIA', 'MIN',
                             'NE', 'NO', 'NYG', 'NYJ', 'LV', 'PHI', 'PIT', 'LAC', 'SEA', 'SF',
                             'LAR', 'TB', 'TEN', 'WAS'], 
                     'FullName':['Arizona','Atlanta','Baltimore','Buffalo','Carolina','Chicago','Cincinnati','Cleveland','Dallas','Denver','Detroit','Green Bay','Houston','Indianapolis','Jacksonville','Kansas City','Miami','Minnesota','New England','New Orleans','N.Y. Giants','N.Y. Jets','Las Vegas','Philadelphia','Pittsburgh','LA Chargers','Seattle','San Francisco','LA Rams','Tampa Bay','Tennessee','Washington']})

#merging the expanded name to referee data
df7000 = pd.merge(df7000, df50[['FullName', 'Team']], on='FullName', how='left')

#staging data
df7000.rename(columns={'Date': 'Day'}, inplace=True)
df7000.rename(columns={'Team': 'HomeTeamName'}, inplace=True)

#changing ref name to a numeric value
ref_mapping = {'Adrian-Hill': 1, 'Alex-Kemp': 2, 'Bill-Vinovich': 3, 'Brad-Allen': 4, 'Brad-Rogers': 5, 'Carl-Cheffers': 6, 'Clay-Martin': 7, 'Clete-Blakeman': 8, 'Craig-Wrolstad': 9, 'Jerome-Boger': 10, 'John-Hussey': 11, 'Ron-Torbert': 12, 'Scott-Novak': 13, 'Shawn-Hochuli': 14, 'Shawn-Smith': 15, 'Tra-Blake': 16, 'Land-Clark':17, 'Alan-Eck':18}
# Note: I removed the duplicate 'Bill-Vinovich': 14 mapping from your original dict.
df7000['Referee'] = df7000['Referee'].replace(ref_mapping)

#changing the date time column to date format allowing to create time based columns
df7000['Day'] = pd.to_datetime(df7000['Day'])

#creating time based features
df7000['month'] = df7000['Day'].dt.month
df7000['year'] = df7000['Day'].dt.year
df7000['dayofyear'] = df7000['Day'].dt.dayofyear

print('---')
print("Final DataFrame head:")
print(df7000.head())

Starting initial class check...
Classes of each table:
['footable']
---
Scraping data for referee: Brad-Allen
Scraping data for referee: Tra-Blake
Scraping data for referee: Clete-Blakeman
Scraping data for referee: Alan-Eck
Request failed for Alan-Eck in 2022 with status code: 404
Scraping data for referee: Jerome-Boger
Request failed for Jerome-Boger in 2023 with status code: 404
Request failed for Jerome-Boger in 2024 with status code: 404
Scraping data for referee: Carl-Cheffers
Scraping data for referee: Land-Clark
Scraping data for referee: Adrian-Hill
Scraping data for referee: Shawn-Hochuli
Scraping data for referee: John-Hussey
Scraping data for referee: Alex-Kemp
Scraping data for referee: Clay-Martin
Scraping data for referee: Scott-Novak
Scraping data for referee: Brad-Rogers
Scraping data for referee: Shawn-Smith
Scraping data for referee: Ron-Torbert
Scraping data for referee: Bill-Vinovich
Scraping data for referee: Craig-Wrolstad
---
Data scraped. Initial DataFrame size

In [ ]:
# Save df7000 as a pickle file
with open('df7000.pickle', 'wb') as f:
    pickle.dump(df7000, f)

In [ ]:
# Load df7000 from the pickle file
with open('df7000.pickle', 'rb') as f:
    df7000 = pickle.load(f)

In [11]:
df7000.head()

,Day,Week,FullName,AwayTeam,Referee,HomeTeamName,month,year,dayofyear
0,2022-09-11,1,Detroit,Philadelphia,4,DET,9,2022,254
1,2022-09-18,2,Dallas,Cincinnati,4,DAL,9,2022,261
2,2022-10-03,4,San Francisco,LA Rams,4,SF,10,2022,276
3,2022-10-09,5,New Orleans,Seattle,4,NO,10,2022,282
4,2022-10-16,6,Kansas City,Buffalo,4,KC,10,2022,289


In [ ]:
###FIVE THIRTY EIGHT

In [36]:
import nfl_data_py as nfl
import pandas as pd
from datetime import date

# Get the most recent seasons (2021 through the current 2025 season)
current_year = date.today().year
years_to_load = list(range(2021, current_year + 1)) 

print(f"Loading NFL Schedule data for seasons: {years_to_load}.")

# --- DATA LOADING (CONFIRMED WORKING) ---
df_nflverse = nfl.import_schedules(years=years_to_load)

# --------------------------------------------------------------------------------
# --- STAGING DATA: Using ONLY CONFIRMED COLUMNS ---
# --------------------------------------------------------------------------------

# 1. Define the base columns that EXIST based on the debugging output
source_cols_to_keep = ['gameday', 'season', 'home_team', 'away_team']

# 2. Column Mapping for renaming
columns_map = {
    'gameday': 'Day',          # Renaming the date column
    'home_team': 'HomeTeamName', 
    'away_team': 'AwayTeamName',
    # Note: 'neutral' and 'elo' columns are omitted as they do not exist in this data set
}

# Select the existing columns and rename them
df1000 = df_nflverse[source_cols_to_keep].rename(columns=columns_map).copy()

# 3. Date Features
df1000['Day'] = pd.to_datetime(df1000['Day'])
df1000['month'] = df1000['Day'].dt.month
df1000['year'] = df1000['Day'].dt.year
df1000['dayofyear'] = df1000['Day'].dt.dayofyear

print("\nData loaded and staged successfully!")
print("NOTE: Elo ratings and the neutral site flag are NOT available in this specific nfl_data_py file.")
df1000.head()

Loading NFL Schedule data for seasons: [2021, 2022, 2023, 2024, 2025].

Data loaded and staged successfully!
NOTE: Elo ratings and the neutral site flag are NOT available in this specific nfl_data_py file.


,Day,season,HomeTeamName,AwayTeamName,month,year,dayofyear
5852,2021-09-09,2021,TB,DAL,9,2021,252
5853,2021-09-12,2021,ATL,PHI,9,2021,255
5854,2021-09-12,2021,BUF,PIT,9,2021,255
5855,2021-09-12,2021,CAR,NYJ,9,2021,255
5856,2021-09-12,2021,CIN,MIN,9,2021,255


In [39]:
df_2022 = df1000[df1000['year'] == 2023].copy()

print(df_2022)

            Day  season HomeTeamName AwayTeamName  month  year  dayofyear
6378 2023-01-01    2022          ATL          ARI      1  2023          1
6379 2023-01-01    2022          DET          CHI      1  2023          1
6380 2023-01-01    2022          HOU          JAX      1  2023          1
6381 2023-01-01    2022           KC          DEN      1  2023          1
6382 2023-01-01    2022           NE          MIA      1  2023          1
...         ...     ...          ...          ...    ...   ...        ...
6672 2023-12-31    2023          WAS           SF     12  2023        365
6673 2023-12-31    2023          SEA          PIT     12  2023        365
6674 2023-12-31    2023          DEN          LAC     12  2023        365
6675 2023-12-31    2023           KC          CIN     12  2023        365
6676 2023-12-31    2023          MIN           GB     12  2023        365

[299 rows x 7 columns]
